# 08 · Role-based recruitment review shortlists

**Decision:** which players warrant further scouting for a stated role and contribution style?

This notebook combines existing striker, CM, winger and movement-xT profiles. It creates up to three distinct review candidates per role, selected through explicit contribution lenses rather than one overall score. These are **retrospective 2022 World Cup review candidates**, not current transfer recommendations or discoveries of financial undervaluation.

Run after notebooks 04–07 on the complete tournament. No downloads are needed. All calculations use the standard library; optional charts use matplotlib. Outputs go to `reports/role_shortlists/`.

This is a prototype of a scouting decision process. Actual recruitment needs a club brief, positional/tactical review, longer-term club performance, video, contract/fee/wage information, injury history, age and availability. Famous international players in this sample are useful method illustrations, not a realistic affordable shortlist.


In [ ]:
import sys
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

assert sys.version_info >= (3, 11), "Use a Python 3.11+ kernel."
PROJECT_OVERRIDE = None  # Optional: Path("/path/to/soccer-recruitment")
if PROJECT_OVERRIDE is not None:
    candidates = [Path(PROJECT_OVERRIDE).expanduser()]
else:
    candidates = []
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([folder, folder / "soccer-recruitment"])
ROOT = next((p.resolve() for p in candidates if (p / "src/soccer/cli.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory, or set PROJECT_OVERRIDE to your complete checkout.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
for folder in ("reports", "data/processed"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("Project:", ROOT.name)

paths = sorted((ROOT / "data/raw/statsbomb/events").glob("*.json"))
if not paths:
    raise FileNotFoundError("Run notebook 01 with acquisition enabled first.")
print("Downloaded event matches:", len(paths))


## 1. Define the brief before looking at selections

Default eligibility: at least 300 elapsed participation minutes, at least 50% of positioned events in the role, complete eligible-movement xT coverage, and at least one scored movement. Role share is event-weighted, not minutes in that position. Features/minutes cover all roles played. CM includes only the midfield positions recorded in notebook 05's manifest, not automatically defensive/attacking midfield or wing backs.

**Selection policy:** choose the highest eligible player for each lens, sequentially, excluding players already selected for that same role. A later lens therefore selects its best **remaining** player, not necessarily its unrestricted leader. Ties prefer more minutes then stable player ID. Striker shot-quality candidates additionally need at least five non-penalty shots. If no candidate qualifies, leave that slot empty; do not lower thresholds automatically. Lens order affects diversification and is saved.

| Role | Three review lenses |
|---|---|
| Striker | Non-penalty chance generation (npxG/90); shot volume (NP shots/90); shot quality (npxG/shot) |
| CM | Forward progression (completed live-ball passes gaining ≥10 x units/90); chance creation (non-penalty shot-linked xG assisted/90); carrying value (carry xT/90) |
| Winger | Carrying value (carry xT/90); chance creation (non-penalty shot-linked xG assisted/90); shooting threat (npxG/90) |

These are different scouting briefs; selection is not a claim of overall superiority. Counts and provider-xG definitions remain as documented in notebooks 04–06. No arbitrary combination of xT, xG and pressures is used.


In [ ]:
import math
import statistics
from datetime import datetime,timezone
MIN_MINUTES=300
MIN_ROLE_SHARE=0.5
MIN_QUALITY_SHOTS=5
REPORT_ROOT=ROOT / "reports/role_shortlists"
REPORT_ROOT.mkdir(parents=True,exist_ok=True)
LENSES={
    "Striker":[("Chance generation","npxg_per90"),("Shot volume","nonpen_shots_per90"),("Shot quality","npxg_per_shot")],
    "CM":[("Forward progression","completed_forward_passes_10u_per90"),("Chance creation","npxg_assisted_per90"),("Carrying value","carry_xt_per90")],
    "Winger":[("Carrying value","carry_xt_per90"),("Chance creation","npxg_assisted_per90"),("Shooting threat","npxg_per90")],
}
METRIC_LABELS={"npxg_per90":"npxG/90","nonpen_shots_per90":"NP shots/90","npxg_per_shot":"npxG/shot",
    "completed_forward_passes_10u_per90":"Forward passes (10 units)/90","npxg_assisted_per90":"NP xG assisted/90","carry_xt_per90":"Carry xT/90"}

def read_csv(path):
    if not path.is_file(): raise FileNotFoundError(f"Missing {path}; run the corresponding earlier notebook.")
    with path.open(newline="",encoding="utf-8") as stream: return list(csv.DictReader(stream))

def numeric_rows(rows):
    text={"player","team","grid","match_id","match_date"}
    for r in rows:
        for k,v in list(r.items()):
            if k in text or v in ("True","False"): continue
            if v=="": r[k]=None
            else:
                try: r[k]=float(v)
                except (ValueError,TypeError): pass
        for k in ("team_id","player_id"):
            if k in r: r[k]=int(r[k])
    return rows

def keyed(rows):
    result={}
    for r in rows:
        key=(r["team_id"],r["player_id"])
        if key in result: raise ValueError(f"Duplicate player-team profile: {key}")
        result[key]=r
    return result

strikers=keyed(numeric_rows(read_csv(ROOT / "reports/striker_all_player_profiles.csv")))
wide_central=keyed(numeric_rows(read_csv(ROOT / "reports/cm_winger_all_profiles.csv")))
xt=keyed(numeric_rows(read_csv(ROOT / "reports/xt_tournament_players.csv")))
match_rows=numeric_rows(read_csv(ROOT / "reports/xt_player_match.csv"))
scenario_rows=numeric_rows(read_csv(ROOT / "reports/xt_robustness/ranking_scenarios.csv"))


In [ ]:
run=json.loads((ROOT / "reports/xt_run_manifest.json").read_text(encoding="utf-8"))
striker_manifest=json.loads((ROOT / "reports/striker_analysis_manifest.json").read_text(encoding="utf-8"))
role_manifest=json.loads((ROOT / "reports/cm_winger_analysis_manifest.json").read_text(encoding="utf-8"))
robustness_manifest=json.loads((ROOT / "reports/xt_robustness/robustness_manifest.json").read_text(encoding="utf-8"))
cohort=set(run["application_match_ids"])
if len(cohort)!=64: raise ValueError("Complete 2022 cohort required.")
for manifest in (striker_manifest,role_manifest):
    if set(manifest["analyzed_match_ids"])!=cohort: raise ValueError("Role and xT cohorts differ.")
    if manifest["source_revisions"]["statsbomb"]!=run["source_revision"]: raise ValueError("Role/source revisions differ.")
if set(robustness_manifest["application_match_ids"])!=cohort or robustness_manifest["source_revision"]!=run["source_revision"]:
    raise ValueError("Robustness and xT runs differ.")
if set(xt)!=set(strikers) or set(xt)!=set(wide_central): raise ValueError("Player rosters differ across source profiles.")
for key,r in xt.items():
    for role_table in (strikers,wide_central):
        if not math.isclose(r["minutes"],role_table[key]["minutes"],rel_tol=1e-9,abs_tol=1e-6):
            raise ValueError(f"Exposure mismatch: {key}")
        if not math.isclose(r["npxg"],role_table[key]["npxg"],abs_tol=1e-8):
            raise ValueError(f"Shooting counts/xG mismatch: {key}")
print("Verified aligned cohorts, source revisions, player IDs, exposure and provider non-penalty xG.")
print("CM and winger definitions:",role_manifest["role_groups"])
print("Striker definitions:",striker_manifest["striker_roles"])


## 2. Build comparable role pools and show uncertainty

Each pool receives its own descriptive percentiles; there are no pooled cross-role percentiles. A percentile describes this selected tournament pool, not a calibrated ability estimate. Ties receive midpoint ranks; if only one player is available, the percentile is missing.

We attach warnings for fewer than 450 minutes, fewer than five appearances, sparse training support, a single match contributing over half of positive movement xT, or a ≥5-place within-role movement-xT/90 rank swing between grid specifications. Thresholds are explicit review heuristics, not statistical tests. Absence of a flag does not establish reliability.

Grid sensitivity concerns movement xT only, not the independent shooting or passing-count lens. Players absent from the saved 300-minute sensitivity scenarios are flagged as incomplete, not assigned an artificial worst rank. Role pools are held fixed while grids change.


In [ ]:
def average_ranks(values):
    order=sorted(values,key=lambda k:-values[k]); ranks={}; i=0
    while i<len(order):
        j=i+1
        while j<len(order) and values[order[j]]==values[order[i]]: j+=1
        for key in order[i:j]: ranks[key]=(i+1+j)/2
        i=j
    return ranks

def within_pool_percentiles(values):
    ranks=average_ranks(values); n=len(values)
    return {key:100*(n-rank)/(n-1) if n>1 else None for key,rank in ranks.items()}
assert within_pool_percentiles({"a":2,"b":2,"c":1})=={"a":75.0,"b":75.0,"c":0.0}
assert within_pool_percentiles({"a":2})=={"a":None}

role_pools={}
all_role_rows=[]
for role in LENSES:
    table=strikers if role=="Striker" else wide_central
    share_field={"Striker":"striker_event_share","CM":"CM_event_share","Winger":"Winger_event_share"}[role]
    pool={}
    for key,source_row in table.items():
        if source_row["minutes"]<MIN_MINUTES or source_row[share_field]<MIN_ROLE_SHARE: continue
        movement=xt[key]
        if movement["unscored_moves"] or not movement["scored_moves"]: continue
        r=dict(source_row)
        for f in ["pass_xt","carry_xt","movement_xt","movement_xt_per90","low_support_moves","scored_moves"]: r[f]=movement[f]
        r["role"]=role; r["role_event_share"]=source_row[share_field]
        r["carry_xt_per90"]=90*r["carry_xt"]/r["minutes"]
        r["pass_xt_per90"]=90*r["pass_xt"]/r["minutes"]
        appearances=[m for m in match_rows if (m["team_id"],m["player_id"])==key and m["minutes"]>0]
        positive=sum(max(m["movement_xt"],0) for m in appearances)
        r["largest_match_positive_xt_share"]=max((max(m["movement_xt"],0) for m in appearances),default=0)/positive if positive else None
        pool[key]=r
    for _,metric in LENSES[role]:
        values={k:r[metric] for k,r in pool.items() if r[metric] is not None}
        for key,pct in within_pool_percentiles(values).items(): pool[key][metric+"_role_percentile"]=pct
    ranks_by_grid={}
    for grid in ["8x6","12x8","16x12"]:
        values={(r["team_id"],r["player_id"]):r["movement_xt_per90"] for r in scenario_rows
                if r["grid"]==grid and r["min_minutes"]==300 and (r["team_id"],r["player_id"]) in pool}
        ranks_by_grid[grid]=average_ranks(values)
    for key,r in pool.items():
        ranks=[v[key] for v in ranks_by_grid.values() if key in v]
        r["grid_rank_scenarios"]=len(ranks)
        r["role_movement_rank_min"]=min(ranks) if ranks else None
        r["role_movement_rank_max"]=max(ranks) if ranks else None
        r["role_movement_rank_swing"]=max(ranks)-min(ranks) if ranks else None
        flags=[]
        if r["minutes"]<450: flags.append("Fewer than 450 elapsed minutes")
        if r["matches"]<5: flags.append("Fewer than five appearances")
        if r["low_support_moves"]: flags.append("Movement xT uses sparse training zones")
        if r["largest_match_positive_xt_share"] is not None and r["largest_match_positive_xt_share"]>.5:
            flags.append("One match contributes over half of positive movement xT")
        if len(ranks)<3: flags.append("Incomplete grid-sensitivity coverage")
        elif r["role_movement_rank_swing"]>=5: flags.append("Within-role movement rank changes by at least five places across grids")
        r["review_flags"]="; ".join(flags) or "No heuristic flags triggered; sample/model limitations remain"
        all_role_rows.append(r)
    role_pools[role]=pool
    print(role,":",len(pool),"eligible player-team profiles")


## 3. Select distinct review candidates with an audit trail

The primary lens determines selection. Other role-relevant measures describe trade-offs; they do not silently enter a weighted score. Positive-statistic requirements prevent assigning a creative or shot-based slot on no observed contribution.

A player can qualify in more than one role if the event-share rule permits it; we prevent duplicates within each role only. All original eligible profiles remain available for comparison, even if not selected.


In [ ]:
shortlist=[]
selection_audit=[]
for role,lenses in LENSES.items():
    selected=set()
    for lens,metric in lenses:
        candidates={k:r for k,r in role_pools[role].items() if k not in selected and r.get(metric) is not None and r[metric]>0}
        if role=="Striker" and lens=="Shot quality":
            candidates={k:r for k,r in candidates.items() if r["nonpen_shots"]>=MIN_QUALITY_SHOTS}
        if not candidates:
            selection_audit.append(dict(role=role,lens=lens,metric=metric,status="No qualifying unselected player",candidate_count=0))
            continue
        key=min(candidates,key=lambda k:(-candidates[k][metric],-candidates[k]["minutes"],k))
        r=candidates[key]; selected.add(key)
        pct=r.get(metric+"_role_percentile")
        quality_note=f"; at least {MIN_QUALITY_SHOTS} NP shots required" if role=="Striker" and lens=="Shot quality" else ""
        record=dict(role=role,lens=lens,metric=metric,metric_label=METRIC_LABELS[metric],team_id=key[0],player_id=key[1],
            player=r["player"],team=r["team"],minutes=r["minutes"],matches=r["matches"],role_event_share=r["role_event_share"],
            metric_value=r[metric],metric_role_percentile=pct,
            movement_xt_per90=r["movement_xt_per90"],pass_xt_per90=r["pass_xt_per90"],carry_xt_per90=r["carry_xt_per90"],
            npxg_per90=r["npxg_per90"],nonpen_shots_per90=r["nonpen_shots_per90"],
            largest_match_positive_xt_share=r["largest_match_positive_xt_share"],role_movement_rank_swing=r["role_movement_rank_swing"],
            review_flags=r["review_flags"],selection_reason=f"Highest {METRIC_LABELS[metric]} among remaining eligible {role} profiles{quality_note}")
        shortlist.append(record)
        selection_audit.append(dict(role=role,lens=lens,metric=metric,status="Selected",candidate_count=len(candidates),player_id=key[1],team_id=key[0]))
        print(f"{role} | {lens} | {r['player']} ({r['team']}) | {METRIC_LABELS[metric]} {r[metric]:.3f} | {r['minutes']:.1f} min")
for role in LENSES:
    keys=[(r["team_id"],r["player_id"]) for r in shortlist if r["role"]==role]
    assert len(keys)==len(set(keys))
assert all(r["minutes"]>=MIN_MINUTES and r["role_event_share"]>=MIN_ROLE_SHARE for r in shortlist)
print("Selection and distinct-player checks passed.")


## 4. Turn each profile into a scouting question

Every candidate needs an evidence statement, a limitation and a next check. Avoid claiming that low pressure counts establish poor work rate or that high pass completion establishes creativity. Neither is supported by raw event counts.

| Lens | Next evidence |
|---|---|
| Shooting / striker quality | Inspect shot locations, supply, set pieces, body parts and a longer club sample. A few good chances can dominate. |
| Forward progression | Inspect pressure, pass difficulty, opponent blocks and the receiver's advantage. The custom 10-unit threshold is not a universal progression definition. |
| Chance creation | Inspect linked shots and open-play/set-piece mix; verify teammates' shot quality and usage. |
| Carrying value | Inspect successful and failed carries/take-ons, opponent shape and video. Successful-movement xT does not directly penalize turnovers. |

World Cup participation and team progression select the sample. Opponent strength, possession share, score state, tactics, and playing outside the nominal role are not controlled here.


In [ ]:
FOLLOW_UP={
    "Chance generation":"Review shot locations, supply and club-season non-penalty xG; separate open-play and set-piece chances.",
    "Shot volume":"Review shot quality, blocked/long-range attempts, team possession and repeatability across a club season.",
    "Shot quality":"Review the individual shots and service patterns; a minimum of five shots remains a very small finishing sample.",
    "Forward progression":"Review under-pressure passes, receiver advantage, opponent block and possession-adjusted progression.",
    "Chance creation":"Inspect linked shots and dead-ball share; test whether creation persists across teammates, opponents and a longer sample.",
    "Carrying value":"Review positive and negative movements, failed take-ons and possession losses with video; event xT misses defenders and off-ball context.",
    "Shooting threat":"Review chance quality and team usage, plus a longer club sample and video of off-ball movement.",
}
for row in shortlist:
    row["next_scouting_check"]=FOLLOW_UP[row["lens"]]
    print("\n",row["player"],"—",row["role"],"/",row["lens"])
    print("Evidence:",row["selection_reason"])
    print("Review:",row["review_flags"])
    print("Next:",row["next_scouting_check"])


## 5. Optional within-role profile charts

Each panel uses one role's eligible pool as the reference. Percentiles describe this sample only, and bars show the separate lens metrics rather than a composite score. Candidate labels include the lens used to select them. These charts deliberately retain trade-offs rather than hiding them in an overall rating.


In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt=None
    print("Charts skipped. Run %pip install matplotlib in a new cell to enable them.")
if plt is not None:
    figures=REPORT_ROOT / "figures"; figures.mkdir(exist_ok=True)
    for role,lenses in LENSES.items():
        candidates=[r for r in shortlist if r["role"]==role]
        if not candidates: continue
        fig,axes=plt.subplots(1,len(candidates),figsize=(5.2*len(candidates),4.8),sharex=True,squeeze=False,constrained_layout=True)
        for ax,selected in zip(axes[0],candidates):
            r=role_pools[role][(selected["team_id"],selected["player_id"])]
            labels=[METRIC_LABELS[m] for _,m in lenses]
            values=[r.get(m+"_role_percentile") for _,m in lenses]
            ax.barh(range(len(labels)),[v if v is not None else 0 for v in values],color=["#246b8e","#40a58c","#d0a347"])
            ax.set_yticks(range(len(labels)),labels,fontsize=8)
            ax.invert_yaxis(); ax.set_xlim(0,105)
            ax.set_xlabel("Percentile within eligible role pool")
            ax.set_title(selected["player"]+"\n"+selected["lens"]+f" | {r['minutes']:.0f} min",fontsize=9)
            for i,v in enumerate(values): ax.text((v if v is not None else 0)+1,i,f"{v:.0f}" if v is not None else "NA",va="center",fontsize=8)
        fig.suptitle(f"{role}: illustrative review candidates; no overall score")
        fig.savefig(figures / f"{role.lower()}_review_profiles.png",dpi=160)
        plt.show()


## 6. Export the evidence and a decision report

The report is a finished draft for review, not a declaration of transfer feasibility. Preserve explicit cohort dates and selection rules in portfolio materials. See the next-scouting-check column for what remains unanswered.


In [ ]:
def write_rows(name,rows):
    if not rows: raise ValueError(f"No rows for {name}")
    columns=list(dict.fromkeys(k for r in rows for k in r))
    with (REPORT_ROOT / name).open("w",newline="",encoding="utf-8") as stream:
        writer=csv.DictWriter(stream,fieldnames=columns); writer.writeheader(); writer.writerows(rows)
write_rows("all_eligible_role_profiles.csv",all_role_rows)
write_rows("review_shortlist.csv",shortlist)
write_rows("selection_audit.csv",selection_audit)
inputs=["striker_all_player_profiles.csv","cm_winger_all_profiles.csv","xt_tournament_players.csv","xt_player_match.csv","xt_robustness/ranking_scenarios.csv"]
import hashlib
manifest=dict(created_at=datetime.now(timezone.utc).isoformat(),cohort_match_ids=sorted(cohort),source_revision=run["source_revision"],
    min_minutes=MIN_MINUTES,min_role_event_share=MIN_ROLE_SHARE,min_shot_quality_attempts=MIN_QUALITY_SHOTS,
    role_definitions=dict(Striker=striker_manifest["striker_roles"],**role_manifest["role_groups"]),
    ordered_lenses=LENSES,selection_policy="Sequential best remaining eligible player for each lens; metric then minutes then player/team ID; no composite score",
    eligible_pool_sizes={role:len(pool) for role,pool in role_pools.items()},
    input_hashes={name:hashlib.sha256((ROOT / "reports" / name).read_bytes()).hexdigest() for name in inputs},
    flags=dict(lower_minutes=450,fewer_matches=5,single_positive_match_share=.5,within_role_grid_rank_swing=5),
    plots_generated=plt is not None,scope="Retrospective tournament scouting-review prototype; no current availability or financial-value claim")
(REPORT_ROOT / "shortlist_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
lines=["# Role-based scouting review: 2022 World Cup", "", "## Decision and scope", "", "Identify players meriting further review for a specified contribution style. This is a retrospective tournament case study, not a current transfer shortlist, causal impact assessment or financial undervaluation claim. StatsBomb Open Data provides the event evidence; movement xT uses a model fitted on 2018 and applied to 2022.", "",f"Eligibility requires at least {MIN_MINUTES} elapsed minutes and {MIN_ROLE_SHARE:.0%} positioned-event share in the role, complete eligible-movement xT coverage and a scored movement. Role assignments are event-based; metrics cover all roles played. Up to three distinct players are selected sequentially per role, using separate lenses rather than an overall score. Striker shot quality additionally requires {MIN_QUALITY_SHOTS} non-penalty shots. Later lenses select the best remaining player; selection order therefore affects the result.", "", "## Role definitions", ""]
for role,roles in manifest["role_definitions"].items(): lines.append(f"- {role}: {', '.join(roles)}.")
for role in LENSES:
    lines += ["",f"## {role} review candidates", "",f"Eligible reference pool: {len(role_pools[role])} player-team profiles.", "", "| Player | Team | Review lens | Primary measure | Minutes |", "|---|---|---|---|---:|"]
    for r in [r for r in shortlist if r["role"]==role]:
        lines.append(f"| {r['player']} | {r['team']} | {r['lens']} | {r['metric_label']}: {r['metric_value']:.3f} | {r['minutes']:.1f} |")
    for r in [r for r in shortlist if r["role"]==role]:
        lines += ["",f"### {r['player']}: {r['lens']}", "",f"**Evidence:** {r['selection_reason']}. Recorded {r['metric_value']:.3f} for the primary measure across {int(r['matches'])} appearances; role-event share {r['role_event_share']:.1%}.", "",f"**Review flags:** {r['review_flags']}.", "",f"**Next scouting check:** {r['next_scouting_check']}"]
lines += ["", "## Interpretation and limits", "", "Role-pool percentiles describe the observed eligible cohort, not player ability. Grid sensitivity applies to movement xT only. Goal/shot and event-count metrics are not validated by that analysis. Temporary injury absences and stoppage time follow the documented elapsed-participation convention. Team possession, tactics, opponents, game state, pass difficulty and off-ball influence remain unadjusted. These filters and heuristic flags do not establish reliability, even when none trigger.", "", "World Cup participation and progression select players and exposure; the cohort cannot represent the full recruitment market. Transfer feasibility needs a real club brief, longer club data, video, age, injuries, fees/wages, contracts and availability. Future predictive claims require a separately specified target and untouched temporal evaluation. Do not tune on the 2022 findings and continue calling that cohort untouched.", "", "Data source: StatsBomb Open Data. Follow the provider README/license and required acknowledgement/logo guidance before publication. Model inspiration and explicit variant assumptions are documented in notebook 06."]
(REPORT_ROOT / "scouting_decision_report.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
print("Saved decision report, candidate tables, full eligible pools and selection audit to",REPORT_ROOT)
